In [2]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from jinja2 import Environment, FileSystemLoader
from tqdm import tqdm
from typing import Callable, List, Dict, Any, Optional, Iterable

# Funtions

In [ ]:
def combine_projects(titles: List[str], summaries: List[str]) -> str:
    """
    Combine two parallel lists—project titles and project summaries—into a
    single structured text block.

    Args:
    titles : List[str]
        A list of project titles. Each element should correspond to the
        project summary at the same index in `summaries`.
    summaries : List[str]
        A list of project summaries. Must be the same length as `titles`.

    Returns:
    str
        A multiline string where each line contains a title followed by its
        corresponding summary. For example:
        "Title 1. Summary 1\nTitle 2. Summary 2"
    """ 

    combined = []

    # Combine each title with its corresponding abstract
    for t, r in zip(titles, summaries):
        combined.append(f"{t}. {r}\n")

    # Join all combined entries into a single multiline string
    return "".join(combined).strip()


def build_text_column(
    df: pd.DataFrame,
    title_col: str = "titulo",
    abs_col: str = "resumen",
    new_col: str = "text_article"
) -> pd.DataFrame:
    """
    Build a new text column by combining all titles and summaries for each
    row in a DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        A DataFrame where each row contains a list of titles and a list of
        summaries corresponding to a researcher or project group.
    title_col : str, optional
        Column name containing lists of titles. Default is "titulo".
    abs_col : str, optional
        Column name containing lists of summaries. Default is "resumen".
    new_col : str, optional
        Name of the column where the combined text will be stored.
        Default is "text_article".

    Returns
    -------
    pd.DataFrame
        The original DataFrame with an added column (`new_col`) containing
        the merged text for each row.

    """

    df[new_col] = df.apply(
        lambda row: combine_projects(row[title_col], row[abs_col]),
        axis=1
    )
    return df

In [5]:
def combine_text_fields(row: pd.Series) -> str | None:
    """
    Combine article and project text fields for a given researcher row.

    Args:
        row: A pandas Series containing 'text_article' and 'text_projects' fields.

    Returns:
        A combined text string if at least one field exists, otherwise None.
    """
    t1 = row['text_article']
    t2 = row['text_projects']

    # Both values missing → return None
    if pd.isna(t1) and pd.isna(t2):
        return None

    # Only article text missing → return project text
    if pd.isna(t1):
        return t2

    # Only project text missing → return article text
    if pd.isna(t2):
        return t1

    # Both fields present → concatenate
    return f"{t1} {t2}"

# Load files and preprocessing

In [7]:
# Load article records from a JSON lines file into a list of dictionaries
df_articles = pd.read_json('articles.json', lines=True)

# Load enrollment records from a JSON lines file into a list of dictionaries
df_enrollment = pd.read_json('enrollment.json', lines=True)

# Load project records from a JSON lines file into a list of dictionaries
df_projects = pd.read_json('projects.json', lines=True)

# Load portfolios records from a JSON lines file into a list of dictionaries
df_portfolios = pd.read_json('portfolios.json', lines=True)

In [ ]:
# Join all valid CUIT
df_cuits = df_enrollment[['cuit']].merge(
    df_portfolios[['cuit']],
    on='cuit',
    how='outer'
)

In [30]:
# Ensure CUIT values are stored as integers for consistent merging and grouping
df_articles['cuit'] = df_articles['cuit'].astype(int)

# Group articles by CUIT and aggregate titles and summaries into lists
df_articles_grouped = df_articles.groupby('cuit').agg({
    'titulo': lambda x: list(x.dropna()),
    'resumen': lambda x: list(x.dropna())
}).reset_index()

# Keep only CUITs that are present in the enrollment dataset
df_articles_grouped = df_articles_grouped[df_articles_grouped['cuit'].isin(df_cuits['cuit'])]

# Build the text representation combining title and summary information
df_articles_grouped = build_text_column(df_articles_grouped)

In [31]:
# Convert project registration dates to datetime format; invalid formats become NaT
df_projects['fecha_alta'] = pd.to_datetime(df_projects['fecha_alta'], errors='coerce')

# Sort projects by registration date in descending order
df_projects = df_projects.sort_values('fecha_alta', ascending=False).reset_index()

# Group projects by CUIT and aggregate multiple text fields into lists
df_projects_grouped = df_projects.groupby('cuit').agg({
    'titulo_proyecto': lambda x: list(x.dropna()),
    'resumen_proyecto': lambda x: list(x.dropna()),
    'palabrasclaves': lambda x: list(x.dropna()),
    'tema_periodo': lambda x: list(x.dropna()),
    'fecha_alta': lambda x: list(x.dropna())
}).reset_index()

# Keep only CUITs present in the enrollment dataset
df_projects_grouped = df_projects_grouped[df_projects_grouped['cuit'].isin(df_cuits['cuit'])]

# Build an aggregated text column from project title and summary information
df_projects_grouped = build_text_column(
    df_projects_grouped,
    'titulo_proyecto',
    'resumen_proyecto',
    'text_projects')

# Create a clean string representation of keywords for each group
df_projects_grouped['text_palabrasclaves'] = (
    df_projects_grouped['palabrasclaves']
    .apply(lambda x: ', '.join(map(str, x)) if isinstance(x, list) else '')
)

In [32]:
# Check the order
df_projects_grouped.loc[1,'fecha_alta']

[Timestamp('2023-07-04 16:04:20'),
 Timestamp('2023-06-10 09:00:59'),
 Timestamp('2023-06-10 09:00:59'),
 Timestamp('2018-05-29 13:36:49'),
 Timestamp('2015-12-10 13:18:51'),
 Timestamp('2014-02-20 09:42:33'),
 Timestamp('2011-11-30 12:58:28')]

In [33]:
# Merge article and project text data by CUIT, keeping all researchers (outer join)
df_researchers = df_articles_grouped[['cuit', 'text_article']].merge(
    df_projects_grouped[['cuit', 'text_projects', 'text_palabrasclaves']],
    on='cuit',
    how='outer'
)

# Generate a unified text field combining article and project information
df_researchers['text'] = df_researchers.apply(combine_text_fields, axis=1)

In [34]:
df_researchers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106 entries, 0 to 105
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   cuit                 106 non-null    int64 
 1   text_article         75 non-null     object
 2   text_projects        106 non-null    object
 3   text_palabrasclaves  106 non-null    object
 4   text                 106 non-null    object
dtypes: int64(1), object(4)
memory usage: 4.3+ KB


In [35]:
# Compute basic statistics on the length of the generated text blocks
min_text_length = df_researchers['text'].apply(len).min()
max_text_length = df_researchers['text'].apply(len).max()
avg_text_length = df_researchers['text'].apply(len).mean()

print(f"Minimum text length: {min_text_length}")
print(f'Maximun text length: {max_text_length}')
print(f'Average text length: {avg_text_length:.2f}')

Minimum text length: 1945
Maximun text length: 114801
Average text length: 24351.34


# Prompts

In [ ]:
from google import genai
from google.genai import types

# Load the API key stored in Google Colab's secret manager
api_key = 'your_KEY'

# Initialize the client for the Gemini Developer API using the retrieved key
client = genai.Client(api_key=api_key)

In [22]:
# Initialize jinja environment
env = Environment(loader=FileSystemLoader("templates"))

In [37]:
def generate_summary_and_keywords(
    info_completa_investigador: str,
    system_instruction_path: Path,
    prompt_path: Path
) -> dict:
    """
    Generate a detailed multi-paragraph professional summary and a list of keywords
    using external .jinja templates for both system instruction and main prompt.

    Parameters
    ----------
    info_completa_investigador : str
        Full concatenated information about the researcher.
    system_instruction_path : Path
        Path object pointing to the .jinja system instruction file.
    prompt_path : Path
        Path object pointing to the .jinja prompt template.

    Returns
    -------
    dict
        Dictionary containing the generated summary and keywords.
    """

    # Validate paths
    if not system_instruction_path.exists():
        raise FileNotFoundError(f"System instruction template not found: {system_instruction_path}")

    if not prompt_path.exists():
        raise FileNotFoundError(f"Prompt template not found: {prompt_path}")


    # Load templates using Jinja2
    env_sys = Environment(loader=FileSystemLoader(system_instruction_path.parent))
    env_prompt = Environment(loader=FileSystemLoader(prompt_path.parent))

    system_instruction_template = env_sys.get_template(system_instruction_path.name)
    prompt_template = env_prompt.get_template(prompt_path.name)

    # Render templates
    system_instruction = system_instruction_template.render()
    prompt = prompt_template.render(info_completa_investigador=info_completa_investigador)

    # Response schema: ensures the model outputs structured JSON
    response_schema = types.Schema(
        type=types.Type.OBJECT,
        properties={
            "resumen_investigador": types.Schema(
                type=types.Type.STRING,
                description="Resumen profesional extenso de varios párrafos (mínimo dos)."
            ),
            "palabras_clave": types.Schema(
                type=types.Type.ARRAY,
                items=types.Schema(type=types.Type.STRING),
                description="Lista de 5 a 10 palabras clave relevantes de la investigación."
            ),
        },
        required=["resumen_investigador", "palabras_clave"]
    )

    # API configuration: higher temperature encourages richer, more detailed content
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0.7,
        response_mime_type="application/json",
        response_schema=response_schema,
    )

    # API call and JSON parsing
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
            config=config,
        )
        return json.loads(response.text)

    except Exception:
        # Fallback structure ensures downstream processing does not break
        print('Error')
        return {
            "resumen_investigador": "Error generating detailed summary.",
            "palabras_clave": ["Error"],
        }


In [ ]:
# Test 1.1: Generate summary and keywords for the first researcher
system1_path = Path('../mapa_ciencia_unc/prompts/profile_summary/system_instruction_1.jinja')
prompt1_path = Path('../mapa_ciencia_unc/prompts/profile_summary/prompt_1.jinja')

generate_summary_and_keywords(df_researchers.loc[0, 'text'], system1_path, prompt1_path)

{'resumen_investigador': 'La trayectoria de este investigador se distingue por un enfoque integral y complejo en el estudio del desarrollo rural, con un énfasis particular en la sustentabilidad y la soberanía alimentaria. Su trabajo se centra en el análisis crítico de los sistemas de producción de alimentos de proximidad en la provincia de Córdoba, contrastando los modelos agroindustriales convencionales con las propuestas agroecológicas. La investigación pone de manifiesto cómo el modelo agroindustrial, intensivo en insumos externos, genera riesgos ambientales y para la salud, y exhibe objetivos económicos a corto plazo que atentan contra la sustentabilidad. En contraposición, el investigador explora las perspectivas y el avance de las experiencias agroecológicas, abogando por un cambio de paradigma productivo hacia formas más resilientes y equitativas, y contribuyendo a la comprensión de la complejidad de la producción alimentaria desde múltiples dimensiones.\n\nUn pilar fundamental 

In [39]:
def process_summaries(
    df: pd.DataFrame,
    system_instruction_path: Path,
    prompt_name: str,
    prompt_path: Path
) -> pd.DataFrame:
    """
    Process each researcher in the DataFrame using a single prompt and store
    the model outputs inside a nested JSON structure ready for MongoDB.

    Notes:
    - Only the API responses are stored. No embeddings are computed here.
    - Can be executed multiple times to append new prompts to the same structure.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain the column 'info_completa_investigador'.
    system_instruction_path : Path
        Path to the .jinja system instruction file.
    prompt_name : str
        Name used as the key under the 'prompts' dictionary.
        Example: "summary_v1"
    prompt_path : Path
        Path to the .jinja prompt template.

    Returns
    -------
    pd.DataFrame
        The same df with an updated 'data' column containing the stored JSON responses.
    """

    # Ensure the column exists or initialize it
    if "data" not in df.columns:
        df["data"] = [{} for _ in range(len(df))]

    # Initialize missing rows
    for i in range(len(df)):
        if not isinstance(df.at[i, "data"], dict):
            df.at[i, "data"] = {}

        if "prompts" not in df.at[i, "data"]:
            df.at[i, "data"]["prompts"] = {}

    # Process rows with a progress bar
    for i, row in tqdm(df.iterrows(), total=len(df), desc=f"Running prompt: {prompt_name}"):

        info = row["text"]

        result = generate_summary_and_keywords(
            info_completa_investigador=info,
            system_instruction_path=system_instruction_path,
            prompt_path=prompt_path
        )

        # Store the result in the nested JSON
        df.at[i, "data"]["prompts"][prompt_name] = result

    return df

In [40]:
system1_path = Path('../mapa_ciencia_unc/prompts/profile_summary/system_instruction_1.jinja')
prompt1_path = Path('../mapa_ciencia_unc/prompts/profile_summary/prompt_1.jinja')

df_researchers = process_summaries(
    df=df_researchers,
    system_instruction_path=system1_path,
    prompt_name="system1_prompt1",
    prompt_path=prompt1_path
)

Running prompt: system1_prompt1: 100%|██████████| 106/106 [22:08<00:00, 12.53s/it]


In [48]:
df_researchers.to_parquet("summariesv1.parquet", index=False, engine="fastparquet")

In [49]:
system2_path = Path('../mapa_ciencia_unc/prompts/profile_summary/system_instruction_2.jinja')
prompt2_path = Path('../mapa_ciencia_unc/prompts/profile_summary/prompt_2.jinja')

df_researchers = process_summaries(
    df=df_researchers,
    system_instruction_path=system2_path,
    prompt_name="system2_prompt2_temp07",
    prompt_path=prompt2_path
)

Running prompt: system2_prompt2_temp07: 100%|██████████| 106/106 [27:07<00:00, 15.35s/it]


In [50]:
df_researchers.to_parquet("summariesv2.parquet", index=False, engine="fastparquet")

In [72]:
system3_path = Path('../mapa_ciencia_unc/prompts/profile_summary/system_instruction_3.jinja')
prompt3_path = Path('../mapa_ciencia_unc/prompts/profile_summary/prompt_3.jinja')

df_researchers = process_summaries(
    df=df_researchers,
    system_instruction_path=system3_path,
    prompt_name="system3_prompt3_temp07",
    prompt_path=prompt3_path
)

Running prompt: system3_prompt3_temp07: 100%|██████████| 106/106 [25:06<00:00, 14.21s/it]


In [76]:
system4_path = Path('../mapa_ciencia_unc/prompts/profile_summary/system_instruction_4.jinja')
prompt2_path = Path('../mapa_ciencia_unc/prompts/profile_summary/prompt_2.jinja')

df_researchers = process_summaries(
    df=df_researchers,
    system_instruction_path=system4_path,
    prompt_name="system4_prompt2_temp07",
    prompt_path=prompt2_path
)

Running prompt: system4_prompt2_temp07:   8%|▊         | 8/106 [06:07<2:31:21, 92.67s/it]

Error


Running prompt: system4_prompt2_temp07: 100%|██████████| 106/106 [33:27<00:00, 18.94s/it] 


In [77]:
df_researchers.to_parquet("summariesv3.parquet", index=False, engine="fastparquet")

In [ ]:
# See which user experienced the failure
df_researchers.loc[7,'data']['prompts']['system4_prompt2_temp07']

{'resumen_investigador': 'Error generating detailed summary.',
 'palabras_clave': ['Error']}

In [ ]:
# Regenerate the summary for this user
result = generate_summary_and_keywords(
    info_completa_investigador=df_researchers.loc[7,'text'],
    system_instruction_path=system4_path,
    prompt_path=prompt2_path
)

df_researchers.loc[7,'data']['prompts']['system4_prompt2_temp07'] = result

In [ ]:
# Check that everything is working properly
df_researchers.loc[7,'data']['prompts']['system4_prompt2_temp07']

{'resumen_investigador': 'La producción científica del investigador se centra en un análisis multifacético de la salud bucal, con una marcada especialización en la genética poblacional de *Streptococcus mutans* y su intrínseca relación con la etiología y epidemiología de la caries dental. Desde una perspectiva metodológica rigurosa, se emplean estudios de genética poblacional para dilucidar la estructura genética (clonal vs. recombinante), el flujo genético, la ascendencia mixta y la historia demográfica de *S. mutans*, utilizando secuencias de genes housekeeping (como *aroE, gltA, gyrA, lepC, tkt, glnA, glk, murI*) y genes de virulencia (*gtfB, spaP*). Las técnicas moleculares incluyen el tipado multilocus de secuencias (MLST), secuenciación de ADN y PCR (incluyendo multiplex PCR para serotipado). Los análisis estadísticos y bioinformáticos son avanzados, abarcando inferencia bayesiana para la estimación de clústeres genéticos y la historia demográfica, así como pruebas de Chi-cuadrad

In [89]:
df_researchers.to_parquet("summariesv3_1.parquet", index=False, engine="fastparquet")

# Embeddings

In [ ]:
def generate_embeddings_for_prompt(
    df: pd.DataFrame,
    prompt_name: str,
    embedding_method_name: str,
    embedding_function: Callable[[str], List[float]],
    rows: Optional[Iterable[int]] = None
) -> pd.DataFrame:
    """
    Generate embeddings for a specific prompt using a chosen embedding method.
    Embeddings are stored inside the 'data' column under:
        data -> embedding_summary -> prompt_name -> embedding_method_name.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing a nested JSON structure under the 'data' column.
        Must contain: data["prompts"][prompt_name]["resumen_investigador"].
    prompt_name : str
        Name of the prompt used when generating summaries/keywords.
    embedding_method_name : str
        Identifier for the embedding model/method (e.g. "openai-large", "bert-base").
    embedding_function : Callable[[str], List[float]]
        Function that takes a string and returns an embedding vector (list of floats).
    rows : Iterable[int], optional
        Specific row indices to process. If None, the function processes all rows.

    Returns
    -------
    pd.DataFrame
        The updated DataFrame with embeddings stored under:
        data["embedding_summary"][prompt_name][embedding_method_name].
    """

    # If no specific rows are provided, process the entire DataFrame
    if rows is None:
        rows = df.index

    # Validate that the dataframe contains the required 'data' column
    if "data" not in df.columns:
        raise ValueError("DataFrame must contain a 'data' column with the JSON structure.")

    # Ensure required keys exist in selected rows
    for i in rows:
        if not isinstance(df.at[i, "data"], dict):
            df.at[i, "data"] = {}

        if "embedding_summary" not in df.at[i, "data"]:
            df.at[i, "data"]["embedding_summary"] = {}

        if prompt_name not in df.at[i, "data"]["embedding_summary"]:
            df.at[i, "data"]["embedding_summary"][prompt_name] = {}

    # Iterate over selected rows using a progress bar
    for i in tqdm(rows, desc=f"Embedding [{embedding_method_name}] for prompt '{prompt_name}'"):

        # Extract the row
        row = df.loc[i]

        # Attempt to extract the text to embed
        try:
            prompt_output = row["data"]["prompts"][prompt_name]["resumen_investigador"]
        except KeyError:
            print(f"Warning: Missing resumen_investigador for row {i}, skipping.")
            continue

        # Compute embedding
        try:
            embedding_vector = embedding_function(prompt_output)
        except Exception as e:
            print(f"Error embedding row {i} with method '{embedding_method_name}': {e}")
            embedding_vector = None

        # Store embedding inside the nested JSON structure
        df.at[i, "data"]["embedding_summary"][prompt_name][embedding_method_name] = embedding_vector

    return df

In [ ]:
def generate_gemini_embedding(
    profile_summary: str,
    output_dim: int = 768
) -> List[float]:
    """
    Generate a semantic embedding using the Gemini model 'gemini-embedding-001'
    and reduce its dimensionality using PCA.

    Parameters
    ----------
    profile_summary : str
        Researcher's summarized profile. It must be a single textual description
        containing background, expertise, and scientific contributions.

    rows : 

    output_dim : int, optional
        Target dimensionality for PCA reduction. Default is 768.

    Returns
    -------
    List[float]
        A single embedding vector with reduced dimensionality.
    """

    try:
        # Configuración para limitar la dimensión de salida
        config = types.EmbedContentConfig(
            output_dimensionality=output_dim
        )

        # Llama al método para generar el embedding
        response = client.models.embed_content(
            model="gemini-embedding-001",
            contents=profile_summary,
            config=config
        )

        # El método embedContent devuelve una lista de objetos de embedding.
        # Como solo pasamos un texto, tomamos el primer (y único) embedding.
        return response.embeddings[0].values

    except Exception as e:
        print(f"Error al generar el embedding: {e}")
        return []

In [ ]:
# Copy the original DataFrame
df_researchers_emb = df_researchers.copy()

In [152]:
# Test 1.1: Generate embedding for the first researcher
user = df_researchers_emb.loc[0,'data']['prompts']['system1_prompt1']['resumen_investigador']

generate_gemini_embedding(user)

[0.013129085,
 0.016281793,
 0.03921327,
 -0.046896316,
 -0.009444237,
 -0.0013313675,
 -0.017094495,
 0.02094679,
 -0.016991308,
 -0.0043000067,
 -0.020608984,
 0.009492011,
 -0.01093993,
 0.065137096,
 0.12921582,
 0.010520229,
 0.018197652,
 -0.01358215,
 0.017056668,
 0.021929167,
 -0.005273874,
 -0.0022222714,
 -0.022104127,
 -0.019570677,
 -0.014763879,
 0.009658671,
 -0.016558565,
 0.0018326074,
 0.030996768,
 -0.008265523,
 -0.008543889,
 0.0608755,
 -0.0119095715,
 0.019583968,
 -0.009756263,
 0.018083258,
 -0.000950118,
 -0.013767334,
 0.018619867,
 0.0072162505,
 0.0072358395,
 -0.029097285,
 -0.034522273,
 0.011529802,
 0.0025629036,
 -0.006564475,
 0.03609642,
 -0.007120372,
 -0.018913904,
 -0.010082129,
 -0.0029637087,
 -0.005740196,
 -0.0034804428,
 -0.16341624,
 0.006795968,
 -0.005076315,
 -0.011585098,
 -0.010937598,
 0.043534108,
 -0.0062587103,
 -0.0069605047,
 0.073701486,
 -0.023380773,
 0.0069295075,
 -0.0037699507,
 -0.012442491,
 0.030750217,
 0.030059127,
 -0.

In [ ]:
df_researchers_emb = generate_embeddings_for_prompt(
    df = df_researchers_emb,
    prompt_name = 'system1_prompt1',
    embedding_method_name = 'gemini_768',
    embedding_function = generate_gemini_embedding,
    rows= range(0,52)
)

In [ ]:
df_researchers_emb = generate_embeddings_for_prompt(
    df = df_researchers_emb,
    prompt_name = 'system1_prompt1',
    embedding_method_name = 'gemini_768',
    embedding_function = generate_gemini_embedding,
    rows = range(52,90)
)

In [ ]:
df_researchers_emb = generate_embeddings_for_prompt(
    df = df_researchers_emb,
    prompt_name = 'system1_prompt1',
    embedding_method_name = 'gemini_768',
    embedding_function = generate_gemini_embedding,
    rows = range(90,106)
)

Embedding [gemini_768] for prompt 'system1_prompt1': 100%|██████████| 16/16 [00:09<00:00,  1.69it/s]


In [ ]:
df_researchers_emb = generate_embeddings_for_prompt(
    df = df_researchers_emb,
    prompt_name = 'system2_prompt2_temp07',
    embedding_method_name = 'gemini_dim768',
    embedding_function = generate_gemini_embedding,
    rows = range(0,40)
)

Embedding [gemini_dim768] for prompt 'system2_prompt2_temp07': 100%|██████████| 40/40 [00:23<00:00,  1.73it/s]


In [ ]:
df_researchers_emb = generate_embeddings_for_prompt(
    df = df_researchers_emb,
    prompt_name = 'system2_prompt2_temp07',
    embedding_method_name = 'gemini_dim768',
    embedding_function = generate_gemini_embedding,
    rows = range(40,80)
)

Embedding [gemini_dim768] for prompt 'system2_prompt2_temp07': 100%|██████████| 40/40 [00:22<00:00,  1.78it/s]


In [ ]:
df_researchers_emb = generate_embeddings_for_prompt(
    df = df_researchers_emb,
    prompt_name = 'system2_prompt2_temp07',
    embedding_method_name = 'gemini_dim768',
    embedding_function = generate_gemini_embedding,
    rows = range(80,106)
)

Embedding [gemini_dim768] for prompt 'system2_prompt2_temp07': 100%|██████████| 26/26 [00:14<00:00,  1.73it/s]


In [ ]:
df_researchers_emb = generate_embeddings_for_prompt(
    df = df_researchers_emb,
    prompt_name = 'system3_prompt3_temp07',
    embedding_method_name = 'gemini_dim768',
    embedding_function = generate_gemini_embedding,
    rows = range(0,40)
)

Embedding [gemini_dim768] for prompt 'system3_prompt3_temp07': 100%|██████████| 40/40 [00:22<00:00,  1.81it/s]


In [ ]:
df_researchers_emb = generate_embeddings_for_prompt(
    df = df_researchers_emb,
    prompt_name = 'system3_prompt3_temp07',
    embedding_method_name = 'gemini_dim768',
    embedding_function = generate_gemini_embedding,
    rows = range(40,80)
)

Embedding [gemini_dim768] for prompt 'system3_prompt3_temp07': 100%|██████████| 40/40 [00:22<00:00,  1.76it/s]


In [ ]:
df_researchers_emb = generate_embeddings_for_prompt(
    df = df_researchers_emb,
    prompt_name = 'system3_prompt3_temp07',
    embedding_method_name = 'gemini_dim768',
    embedding_function = generate_gemini_embedding,
    rows = range(80,106)
)

Embedding [gemini_dim768] for prompt 'system3_prompt3_temp07': 100%|██████████| 26/26 [00:14<00:00,  1.82it/s]


In [ ]:
df_researchers_emb = generate_embeddings_for_prompt(
    df = df_researchers_emb,
    prompt_name = 'system4_prompt2_temp07',
    embedding_method_name = 'gemini_dim768',
    embedding_function = generate_gemini_embedding,
    rows = range(0,40)
)

Embedding [gemini_dim768] for prompt 'system4_prompt2_temp07': 100%|██████████| 40/40 [00:22<00:00,  1.80it/s]


In [ ]:
df_researchers_emb = generate_embeddings_for_prompt(
    df = df_researchers_emb,
    prompt_name = 'system4_prompt2_temp07',
    embedding_method_name = 'gemini_dim768',
    embedding_function = generate_gemini_embedding,
    rows = range(40,80)
)

Embedding [gemini_dim768] for prompt 'system4_prompt2_temp07': 100%|██████████| 40/40 [00:22<00:00,  1.75it/s]


In [ ]:
df_researchers_emb = generate_embeddings_for_prompt(
    df = df_researchers_emb,
    prompt_name = 'system4_prompt2_temp07',
    embedding_method_name = 'gemini_dim768',
    embedding_function = generate_gemini_embedding,
    rows = range(80,106)
)

Embedding [gemini_dim768] for prompt 'system4_prompt2_temp07': 100%|██████████| 26/26 [00:14<00:00,  1.84it/s]


In [ ]:
df_researchers_emb.to_parquet("summaries_embeddingsv1.parquet", index=False, engine="fastparquet")